# Aula 4 - Funções, *args/**kwargs, Classes e Bibliotecas Nativas


Nesta aula veremos como organizar melhor programas em Python. 

Abordaremos definição de funções, formas flexíveis de passar argumentos com *args e **kwargs, e o uso de tuplas e dicionários no desempacotamento de parâmetros.

Também introduziremos classes e objetos, destacando seus atributos, métodos e a ideia de invariantes. 

Por fim, veremos exemplos de uso de bibliotecas nativas do Python, como math e time.

---

### Estrutura da aula

- **1 Funções** 
- **2 \*args e \*\*kwargs** 
- **3 Desempacotamento** de tuplas e dicionários (`*` e `**`)
- **4 Wrappers e decorators** 
- **5 Classes**
- **6 Difereença de classe e objeto** 
- **7 Invariantes**
- **8 Classes** 
- **9 Bibliotecas nativas** (math, time, functools, datetime…)



## 0. Setup rápido (imports que vamos usar)


In [10]:
import time
import math
from functools import wraps
from dataclasses import dataclass


## 1. Definição de funções

Uma **função** é um bloco de código com nome, que pode receber entradas (parâmetros) e devolver um resultado (`return`).

Por que isso importa?
- Evita repetição
- Facilita testes
- Deixa o código mais legível
- Permite construir “peças” que você combina depois

### Sintaxe
- `def nome(parametros):`
- corpo identado
- `return` devolve o resultado (se não tiver `return`, o Python devolve `None`)


In [ ]:
def calcular_retorno(preco_inicial: float, preco_final: float) -> float:
    # Retorno percentual simples: (P1 - P0) / P0
    return (preco_final - preco_inicial) / preco_inicial
calcular_retorno(100, 120)


0.9

### Parâmetros com valor padrão

Valores padrão deixam sua função mais fácil de usar sem perder flexibilidade.
Aqui, `tempo=1` significa que, se você não informar `tempo`, ele vale 1.


In [ ]:
def calcular_montante(capital: float, taxa: float, tempo: int = 1) -> float:
    # Montante em juros compostos: M = C * (1 + i) ** t
    return capital * ((1 + taxa) ** tempo)

calcular_montante(1000, 0.10, 1)


1210.0000000000002

### Funções são objetos (importantíssimo)

Em Python, funções são **objetos**:
- podem ser guardadas em variáveis
- passadas como argumento
- retornadas por outras funções

Isso aparece direto em `sorted(key=...)`, `DataFrame.apply(...)`, `map`, `filter`, etc.


In [30]:
f = calcular_retorno  # agora 'f' aponta para a função
f(200, 260)


0.3

## 2. *args e **kwargs

Quando você não sabe quantos argumentos vão ser passados (ou quer criar uma API flexível), você usa:

- `*args`  → captura **argumentos posicionais** extras em uma **tupla**
- `**kwargs` → captura **argumentos nomeados** extras em um **dicionário**

Isso é muito usado em bibliotecas para permitir opções extras sem “quebrar” a função.


### 2.1 *args (posicionais variáveis)


In [ ]:
def media(*valores: float) -> float:
    # *valores vira uma tupla.
    return sum(valores) / len(valores)

media(10, 20, 30, 40, 70, 70, 70, 70, 70, 70, 70, 70)


55.0

### 2.2 **kwargs (nomeados variáveis)


In [ ]:
def mostrar_parametros(**kwargs):
    # **kwargs vira um dict.
    for chave, valor in kwargs.items():
        print(f"{chave} -> {valor}")

mostrar_parametros(crescimento=0.05, taxa=0.1, moeda="BRL")


crescimento -> 0.05
taxa -> 0.1
moeda -> BRL


### 2.3 Exemplo prático: função configurável via kwargs

Vamos criar uma função de “simulação” que aceita opções adicionais via `kwargs`.
Padrão comum: alguns parâmetros fixos + várias opções extras (configs).


In [93]:
def simular_montante(capital: float, taxa: float, tempo: int, **kwargs) -> float:
    # Opções (todas opcionais):
    # - aporte_por_periodo: float (default 0)
    # - imposto: float (default 0) -> imposto sobre o ganho (simplificado)
    # - arredondar: int (default None) -> casas decimais

    aporte_por_periodo = kwargs.get("aporte_por_periodo", 0.0)
    imposto = kwargs.get("imposto", 0.0)
    arredondar = kwargs.get("arredondar", None)

    montante = capital
    ganho = 0
    montante_liquido = 0

    for _ in range(tempo):
        D_0 = montante
        # simplificação didática: taxa anual aplicada por período + aportes mensais do ano
        montante = montante * (1 + taxa) + aporte_por_periodo

        ganho += ( montante - D_0 )
        montante_liquido = montante - imposto * ganho

    if arredondar is not None:
        return round(montante_liquido, arredondar)
    
    return montante_liquido

simular_montante(10_000, 0.10562624, 2, aporte_por_periodo=200, imposto=0.15, arredondar=0)


12248.0

## 3. Passando tuplas e dicionários em funções (`*` e `**`)

O `*` e o `**` também servem para **desempacotar** dados ao chamar uma função:

- `func(*tupla_ou_lista)` passa cada elemento como argumento posicional
- `func(**dict)` passa cada par chave/valor como argumento nomeado


In [95]:
valores = ( 100, 150 )

calcular_retorno(*valores)


0.5

In [97]:
params = {"capital": 1000, "taxa": 0.1, "tempo": 3}

calcular_montante(**params)


1331.0000000000005

### 3.1 Misturando: fixos + *args + **kwargs


In [52]:
def registrar_trades(ativo: str, *precos: float, **meta):
    print(f"Ativo: {ativo}")
    print(f"Preços: {precos}")
    print(f"Meta: {meta}")

registrar_trades("PETR4", 30.1, 30.2, 30.0, origem="B3", trader="Tiago")


Ativo: PETR4
Preços: (30.1, 30.2, 30.0)
Meta: {'origem': 'B3', 'trader': 'Tiago'}


## 4. Tempo de execução com kwargs e Decorators

Medir tempo de execução ajuda a comparar abordagens e encontrar gargalos.
Vamos evoluir em 3 níveis: wrapper -> decorator -> decorator com argumento.


### 4.1 Wrapper genérico: `executar_com_tempo(func, *args, **kwargs)`


In [57]:
def executar_com_tempo(func, *args, **kwargs):
    inicio = time.perf_counter()
    resultado = func(*args, **kwargs)
    fim = time.perf_counter()
    print(f"Tempo de execução de {func.__name__}: {fim - inicio:.6f}s")
    return resultado

def potencia(base: float, expoente: int = 2) -> float:
    return base ** expoente

executar_com_tempo(potencia, 10, expoente=6)


Tempo de execução de potencia: 0.000008s


1000000

### 4.2 Decorator: medindo tempo automaticamente


In [62]:
def medir_tempo(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = func(*args, **kwargs)
        fim = time.perf_counter()
        print(f"Tempo de execução de {func.__name__}: {fim - inicio:.6f}s")
        return resultado
    return wrapper

@medir_tempo
def soma_grande(n: int) -> int:
    total = 0
    for i in range(n):
        total += i
    return total

@medir_tempo
def multiplicacao (num1, num2):
    return num1 * num2

multiplicacao(2_000_000, 1_000)


Tempo de execução de multiplicacao: 0.000003s


2000000000

### 4.3 Decorator com argumento: logar apenas se for lento


In [64]:
def log_se_lento(limite_segundos: float = 0.01):

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            inicio = time.perf_counter()
            resultado = func(*args, **kwargs)

            fim = time.perf_counter()

            dt = fim - inicio
            if dt > limite_segundos:
                print(f"[SLOW] {func.__name__} levou {dt:.6f}s (limite={limite_segundos}s)")
            return resultado
        return wrapper
    return decorator

@log_se_lento(limite_segundos=0.5)
def ordenar_lista(n: int):
    xs = list(range(n, 0, -1))
    xs.sort()
    return xs[0], xs[-1]

ordenar_lista(200_000)


(1, 200000)

## 5. Definição de classes

Classes modelam entidades e encapsulam:
- atributos (dados)
- métodos (comportamentos)


In [65]:
class Ativo:
    def __init__(self, nome: str, preco: float):
        self.nome = nome
        self.preco = preco

    def atualizar_preco(self, novo_preco: float) -> None:
        self.preco = novo_preco

    def retorno(self, preco_inicial: float) -> float:
        return (self.preco - preco_inicial) / preco_inicial

acao = Ativo("PETR4", 30.0)

acao.retorno(25.0)


0.2

## 6. Diferença entre classe e objeto

- Classe: molde
- Objeto: instância do molde


In [ ]:
a1 = Ativo("VALE3", 62.0)
a2 = Ativo("VALE3", 62.0)

a1 is a2


(False, True, True)

## 7. Invariantes com @property

Invariantes são regras que mantêm o objeto em estado válido.
Aqui: preço não pode ser negativo.


In [70]:
class AtivoSeguro:
    def __init__(self, nome: str, preco: float):
        self.nome = nome
        self._preco = 0.0
        self.preco = preco

    @property
    def preco(self) -> float:
        return self._preco

    @preco.setter
    def preco(self, novo_preco: float) -> None:
        if novo_preco < 0:
            raise ValueError("Preço não pode ser negativo")
        self._preco = float(novo_preco)

    def retorno(self, preco_inicial: float) -> float:
        return (self.preco - preco_inicial) / preco_inicial

seguro = AtivoSeguro("ITUB4", 32.5)
print(seguro.preco)
seguro.preco = 33.0
print(seguro.preco)



32.5
33.0


### 7.1 Onde isso aparece em bibliotecas?

Bibliotecas grandes precisam validar entradas e manter consistência interna:
- DataFrame: colunas com mesmo tamanho
- NumPy: shape e dtype consistentes
- APIs: parâmetros válidos evitam bugs silenciosos


## 8. Dataclasses
Um **boilerplate** é um bloco de código, estrutura de projeto ou template reutilizável que resolve a parte “padrão” de uma tarefa.

Ele normalmente:

1. reduz repetição;
2. padroniza a organização do código;
3. acelera o desenvolvimento;
4. diminui erros em tarefas recorrentes.

### 8.1 Boilerplate de tratamento de erro

Um padrão muito comum é encapsular operações arriscadas com `try/except`.


In [72]:
def dividir(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Erro: divisão por zero"

print(dividir(10, 2))
print(dividir(10, 0))
print(dividir(7, 0))
print(dividir(10, 4))

5.0
Erro: divisão por zero
Erro: divisão por zero
2.5


## 9. Bibliotecas nativas (math, time, datetime)

Sem instalar nada, dá pra resolver muita coisa no dia a dia.


### 9.1 math


In [ ]:
math.sqrt(81), math.log(100, 10)


(9.0, 2.0)

### 9.2 time


In [ ]:
inicio = time.perf_counter()
time.sleep(0.2)
fim = time.perf_counter()
fim - inicio


5299.857667211


### 9.3 datetime (bônus)


In [78]:
import datetime as dt
agora = dt.datetime.now()
print(agora)
print(agora.strftime("%d-%m-%Y %H:%M:%S"))


2026-04-30 19:07:55.871014
30-04-2026 19:07:55


## Encerramento

Você já tem a base para entender APIs flexíveis e OOP em bibliotecas reais.
Na próxima etapa, isso conversa diretamente com estruturas como `DataFrame`, `Series` e arrays vetorizados.
